# Volume 1, Chapter 3: Congruences (Python companion)

Exercises first, solutions at the end. The computer-science thread is **RSA**, built end to end from this chapter:
Euler's theorem makes it work, the extended Euclidean algorithm makes the keys, and the Chinese remainder theorem makes it fast.
We also break it when it is badly implemented.

In [ ]:
import math
import random
import time
import numpy as np
import matplotlib.pyplot as plt
from sympy import randprime, nextprime, isprime
# Reproducible teaching examples; these seeds are not for cryptographic keys.
import random
from sympy.core.random import seed as seed_sympy
random.seed(10103)
seed_sympy(20103)


## Before the experiment: An inverse is a contract

Write down the assumptions before each algorithm: positive modulus; nonnegative exponent; coprime CRT moduli. **Cryptography laboratory:** these are arithmetic demonstrations, using deterministic randomness and textbook RSA. They omit secure padding (OAEP/PSS), side-channel defenses, and production key handling. Use a vetted cryptographic library for real secrets.

**How to use this notebook.** Try the exercise cells first; they are intentionally blank. Then run the solutions in order in a fresh kernel. Assertions check exact identities or a stated finite range; plots and random trials supply evidence, not proofs.


## Exercises

**P3.1 (★) Fast exponentiation.** Implement `power_mod(a, e, n)` by square-and-multiply (scan the bits of $e$). How many multiplications does it use for a 2048-bit exponent? Check Exercises 3.1 and 3.7 with it.

In [ ]:
# Your code here


**P3.2 (★) A general CRT solver.** Write `crt(residues, moduli)` for pairwise coprime moduli using the formula of Theorem 3.5 repeatedly, and solve Sunzi's problem (Exercise 3.2).

In [ ]:
# Your code here


**P3.3 (★★) CS application: RSA.** 
1. *Key generation:* pick random primes $p, q$ of 1024 bits, $n = pq$, $e = 65537$, and $d = e^{-1} \bmod \varphi(n)$.
2. *Encryption* $c = m^e \bmod n$ and *decryption* $m = c^d \bmod n$. Prove that decryption works for every $m$ coprime to $n$, using Euler's theorem (in fact it works for all $m$, by Korselt's argument).
3. *Signatures:* $s = h^d \bmod n$ is checked by $s^e \equiv h$.
4. *CRT speed-up:* compute $m_p = c^{d \bmod (p-1)} \bmod p$ and $m_q$ similarly, then combine with the CRT. Time both decryptions.

In [ ]:
# Your code here


**P3.4 (★★) Breaking bad RSA: close primes.** If $|p - q|$ is small, then $n = a^2 - b^2$ with $a = \frac{p+q}2$ just above $\sqrt n$ and $b = \frac{q-p}2$ small (Fermat's factorisation method). Generate a 2048-bit $n$ with $q$ = the next prime after $p$, and factor it instantly.

In [ ]:
# Your code here


**P3.5 (★★) Breaking bad RSA: shared primes.** Two different public keys that accidentally share a prime are both broken by a single gcd. Simulate 200 keys where two share a prime, and find them. (In 2012, researchers found tens of thousands of such keys on the internet.)

In [ ]:
# Your code here


**P3.6 (★★★) Lucas' theorem, drawn.** Plot Pascal's triangle modulo 3 and modulo 5 (128 rows), coloured by residue, and verify Lucas' theorem numerically for all $n < 300$ and $p \in \{2, 3, 5, 7\}$.

In [ ]:
# Your code here


---
## Solutions

### P3.1

In [ ]:
def power_mod(a, e, n):
    if e < 0 or n < 1:
        raise ValueError("require e >= 0 and n >= 1")
    result, a, mults = 1 % n, a % n, 0
    while e:
        if e & 1:
            result = result * a % n; mults += 1
        a = a * a % n; mults += 1
        e >>= 1
    return result, mults

print("7^2026 mod 100 =", power_mod(7, 2026, 100)[0])
print("3^(3^27) mod 100 =", power_mod(3, 3**27, 100)[0])
e = random.getrandbits(2048) | (1 << 2047)
print("multiplications for a 2048-bit exponent:", power_mod(5, e, 10**9 + 7)[1], "(at most 2 x 2048)")

### P3.2

In [ ]:
def crt(residues, moduli):
    if len(residues) != len(moduli) or any(n < 2 for n in moduli):
        raise ValueError("use equally long lists and moduli >= 2")
    x, m = 0, 1
    for a, n in zip(residues, moduli):
        # combine x mod m with a mod n
        u = pow(m, -1, n)
        x = x + m * ((a - x) * u % n)
        m *= n
    return x % m, m

print("Sunzi:", crt([2, 3, 2], [3, 5, 7]))

### P3.3

In [ ]:
def keygen(bits=1024):
    e = 65537
    while True:
        p = randprime(2 ** (bits - 1), 2**bits)
        q = randprime(2 ** (bits - 1), 2**bits)
        if p != q and math.gcd(e, (p - 1) * (q - 1)) == 1:
            break
    d = pow(e, -1, (p - 1) * (q - 1))
    return (p * q, e), (p, q, d)

(n, e), (p, q, d) = keygen()
message = b"Order and Chance"
m = int.from_bytes(message, "big")
assert m < n
c = pow(m, e, n)
assert pow(c, d, n) == m
print("decrypted:", pow(c, d, n).to_bytes(len(message), "big"))

h = int.from_bytes(__import__("hashlib").sha256(b"I approve this chapter").digest(), "big")
s = pow(h, d, n)
print("signature valid:", pow(s, e, n) == h)

dp, dq, q_inv = d % (p - 1), d % (q - 1), pow(q, -1, p)
def decrypt_crt(c):
    mp, mq = pow(c, dp, p), pow(c, dq, q)
    return mq + q * ((mp - mq) * q_inv % p)       # Garner's form of the CRT
assert decrypt_crt(c) == m

t0 = time.perf_counter(); [pow(c, d, n) for _ in range(50)]; t1 = time.perf_counter(); [decrypt_crt(c) for _ in range(50)]; t2 = time.perf_counter()
print(f"plain decryption {1000 * (t1 - t0) / 50:.1f} ms, CRT decryption {1000 * (t2 - t1) / 50:.1f} ms")
# CRT also explains decryption of messages not coprime to n.
for test_message in [0, 1, p, q, n - 1]:
    assert pow(pow(test_message, e, n), d, n) == test_message


**Why it works.** $ed \equiv 1 \pmod{\varphi(n)}$, so $ed = 1 + k\varphi(n)$ and, for $\gcd(m,n) = 1$, $m^{ed} = m\,(m^{\varphi(n)})^k \equiv m$ by Euler. For general $m$, argue modulo $p$ and $q$ separately as in Exercise 3.8. **Why CRT is faster:** with schoolbook integer multiplication, exponentiation costs roughly (exponent bits) × (modulus bits)². Two half-size exponentiations then cost about one quarter as much. Real big-integer libraries use faster multiplication, so measure the gain instead of assuming a universal factor of four.

### P3.4

In [ ]:
p = randprime(2**1023, 2**1024)
q = nextprime(p)
n = p * q
t = time.perf_counter()
a = math.isqrt(n)
if a * a < n:
    a += 1
while True:
    b2 = a * a - n
    b = math.isqrt(b2)
    if b * b == b2:
        break
    a += 1
print(f"factored a 2048-bit n in {1000 * (time.perf_counter() - t):.2f} ms:", a - b == p and a + b == q)

### P3.5

In [ ]:
shared = randprime(2**255, 2**256)
moduli = [randprime(2**255, 2**256) * randprime(2**255, 2**256) for _ in range(198)]
moduli += [shared * randprime(2**255, 2**256), shared * randprime(2**255, 2**256)]
random.shuffle(moduli)
broken = [(i, j) for i in range(len(moduli)) for j in range(i + 1, len(moduli)) if math.gcd(moduli[i], moduli[j]) > 1]
print("keys sharing a prime:", broken, " recovered prime correct:", math.gcd(moduli[broken[0][0]], moduli[broken[0][1]]) == shared)

Real attacks on millions of keys replace the $O(k^2)$ pairwise gcds by a *product tree* and *remainder tree*, computing all gcds in quasi-linear time.

### P3.6

In [ ]:
def binom_mod_lucas(n, k, p):
    r = 1
    while n or k:
        ni, ki = n % p, k % p
        if ki > ni:
            return 0
        r = r * math.comb(ni, ki) % p
        n //= p; k //= p
    return r

for p in [2, 3, 5, 7]:
    for n in range(300):
        for k in range(n + 1):
            assert math.comb(n, k) % p == binom_mod_lucas(n, k, p)
print("Lucas' theorem verified")

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, p in zip(axes, [3, 5]):
    grid = np.full((128, 128), np.nan)
    for n in range(128):
        for k in range(n + 1):
            grid[n, k] = math.comb(n, k) % p
    ax.imshow(grid, cmap="viridis", interpolation="nearest"); ax.set_title(f"Pascal's triangle mod {p}"); ax.axis("off")
plt.show()

## One more question before closing

Why do 0, p, and q decrypt correctly although Euler’s theorem cannot be applied to them modulo pq? Prove the two prime-modulus statements separately.

Write one sentence identifying what was proved in the chapter, what was checked exactly by the code, and what remains a conjecture or a heuristic.
